In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten, Reshape, Conv2D, Conv2DTranspose
from tensorflow.keras import layers

In [ ]:
train_data = pd.read_csv('./Kannada-MNIST/train.csv')
test_data = pd.read_csv('./Kannada-MNIST/test.csv')

In [ ]:
train_data

In [ ]:
train_images = train_data.iloc[:, 1:].values.reshape(-1, 28, 28, 1).astype('float32') / 255.0
test_images = test_data.iloc[:, 1:].values.reshape(-1, 28, 28, 1).astype('float32') / 255.0

In [ ]:
test_images.shape

In [ ]:
# Encoder
inputs = Input(shape=(28, 28, 1))
x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
x = layers.MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(16, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2), padding='same')(x)

# Decoder
x = Conv2DTranspose(16, (3, 3), strides=2, activation='relu', padding='same')(x)
x = Conv2DTranspose(32, (3, 3), strides=2, activation='relu', padding='same')(x)
outputs = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

# Autoencoder model
autoencoder = Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
autoencoder.summary()


In [ ]:
autoencoder.fit(train_images, train_images, epochs=10, batch_size=32, shuffle=True, validation_data=(test_images, test_images))

In [ ]:
def denoise_images(img):

    denoised_images = img + 0.1 * tf.random.normal(shape=img.shape)
    return denoised_images

In [ ]:
# Predict (Denoise)
denoised_images = autoencoder.predict(test_images)

In [ ]:
n = 10
plt.figure(figsize=(20, 6))
for i in range(n):

    ax = plt.subplot(3, n, i + 1)
    plt.imshow(test_images[i].reshape(28, 28), cmap='gray')
    plt.title("Original")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(denoise_images(test_images[i].reshape(28, 28)), cmap='gray')
    plt.title("Noised")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(denoised_images[i].reshape(28, 28), cmap='gray')
    plt.title("Denoised")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
plt.savefig("example.png")
plt.show()